# C6 example 3/4: `WETensorProduct` with external circular harmonics

This architecture makes a message geometry-conditioned. The node still contains exactly three 3D vectors and two scalars, packed as $3E_1\oplus5A$. Separately, an edge has displacement $r_{ij}=p_j-p_i$; it is filter geometry and is not part of the node representation. For C6, $(r_x,r_y)$ rotates while $r_z$ is invariant. If there is no edge/filter geometry, `WELinear` or the invariant-context `TensorProduct` example is the appropriate construction.

`WETensorProduct` expects filter features to be computed externally, so this notebook performs

$$r_{xy}\xrightarrow{\operatorname{atan2}}\theta\xrightarrow{Y_{C_6}}Y(\theta)\xrightarrow{\mathrm{WETP}}\text{features}.$$

The default C6 circular bandlimit is $L_{full}=\lfloor6/2\rfloor=3$. Radius and $r_z$ are invariant inputs to small radial networks that produce the per-sample reduced weights. The radial-network parameters are shared over all nodes/messages, although their numerical outputs may depend on geometry.

In [ ]:
import torch
from we3nn import CircularHarmonics, CyclicGroup, nn

torch.manual_seed(7)
torch.set_printoptions(precision=5, sci_mode=False)
G = CyclicGroup(6)
A = G.trivial_representation
E1 = G.standard_representation
regular = G.regular_representation()
input_rep = 3 * E1 + 5 * A
hidden_rep = 2 * regular
output_rep = E1 + 4 * A
harmonics = CircularHarmonics(G)  # max_frequency=None -> floor(6/2)=3
filter_rep = harmonics.rep_out
print('harmonic frequencies: 0..', harmonics.max_frequency)
print('filter representation:', filter_rep.name)
print('dimensions:', input_rep.size, 'x', filter_rep.size, '->', hidden_rep.size, '->', output_rep.size)

In [ ]:
def pack_input(vectors, scalars):
    xy = vectors[..., :, :2].reshape(*vectors.shape[:-2], 6)
    return torch.cat((xy, vectors[..., :, 2], scalars), dim=-1)

def unpack_input(x):
    xy = x[..., :6].reshape(*x.shape[:-1], 3, 2)
    return torch.cat((xy, x[..., 6:9].unsqueeze(-1)), dim=-1), x[..., 9:11]

def unpack_output(y):
    return torch.cat((y[..., :2], y[..., 2:3]), dim=-1), y[..., 3:6]

def rotate_points(points, element):
    Rxy = E1(element).to(device=points.device, dtype=points.dtype)
    xy = points[..., :2] @ Rxy.T
    return torch.cat((xy, points[..., 2:3]), dim=-1)

vectors = torch.tensor([[[1.0, 0.2, -0.4], [-0.3, 0.8, 1.2], [0.5, -0.7, 0.1]]])
scalars = torch.tensor([[0.6, -1.1]])
# This is edge geometry r_ij, not a fourth node-feature vector.
edge_displacement = torch.tensor([[0.8, 0.35, -0.2]])
x = nn.RepresentationTensor(pack_input(vectors, scalars), input_rep)

angle = torch.atan2(edge_displacement[..., 1], edge_displacement[..., 0])
Y = harmonics(angle)  # equivalently: harmonics.from_vectors(edge_displacement[..., :2])
print('edge displacement r_ij:', edge_displacement)
print('angle:', angle)
print('external circular harmonics Y(theta):', Y)

## Step 1: distinguish node features from edge geometry

The three colored arrows are node features. The dashed black arrow is $r_{ij}=p_j-p_i$: it tells the filter where the neighboring/source node lies relative to the receiving node. It is not packed into `x` and is not a learned or additional node vector.

In [ ]:
import matplotlib.pyplot as plt

fig_input = plt.figure(figsize=(6, 5), constrained_layout=True)
ax = fig_input.add_subplot(111, projection='3d')
for index, vector in enumerate(vectors[0]):
    ax.quiver(0, 0, 0, *vector.tolist(), color=f'C{index}', linewidth=2, label=f'node feature v{index+1}')
ax.quiver(0, 0, 0, *edge_displacement[0].tolist(), color='black', linestyle='--', linewidth=2, label='edge displacement r_ij')
limit = 1.15 * torch.cat((vectors[0], edge_displacement)).abs().max().item()
ax.set(xlim=(-limit, limit), ylim=(-limit, limit), zlim=(-limit, limit), xlabel='x', ylabel='y', zlabel='z', title='Node features versus filter geometry')
ax.set_box_aspect((1, 1, 1)); ax.legend(fontsize=8)
print('scalar node features:', scalars[0].tolist())
plt.show()

## Step 2: see the circular harmonics on their domain

A circular harmonic is a scalar component of a function on $S^1$. Each polar panel below evaluates one component continuously around the unit circle. The radius is $1+0.35Y(\theta)/\max|Y|$: values outside the dashed unit circle are positive and values inside are negative. Dots mark the six orientations in the C6 orbit of the chosen edge.

In [ ]:
import math
import matplotlib.pyplot as plt

theta_grid = torch.linspace(0.0, 2.0 * math.pi, 361)
Y_circle = harmonics(theta_grid).detach()
orbit_angles = angle[0] + torch.arange(6) * (2.0 * math.pi / 6.0)
Y_orbit = harmonics(orbit_angles).detach()

harmonic_labels = []
for frequency, mode in harmonics._layout:
    if mode == 'pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'sin({frequency}θ)'))
    elif mode == 'conjugate_pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'-sin({frequency}θ)'))
    else:
        harmonic_labels.append(f'{mode}({frequency}θ)')

fig_circle, axes = plt.subplots(2, 4, figsize=(13, 7), subplot_kw={'projection': 'polar'}, constrained_layout=True)
for component, (ax, label) in enumerate(zip(axes.flat, harmonic_labels)):
    scale = Y_circle[:, component].abs().max().clamp_min(1e-8)
    radius = 1.0 + 0.35 * Y_circle[:, component] / scale
    orbit_radius = 1.0 + 0.35 * Y_orbit[:, component] / scale
    ax.plot(theta_grid, torch.ones_like(theta_grid), color='gray', linestyle='--', linewidth=0.8)
    ax.plot(theta_grid, radius, linewidth=2)
    ax.scatter(orbit_angles, orbit_radius, c=torch.arange(6), cmap='hsv', s=30, zorder=3)
    ax.set_ylim(0.6, 1.4); ax.set_yticklabels([]); ax.set_title(label)
axes.flat[-1].set_axis_off()
fig_circle.suptitle('Every circular-harmonic component used by C6 (frequencies 0, 1, 2, 3)', fontsize=14)
plt.show()

## Step 3: build the externally filtered architecture

For each layer, the finite-group coupling tensors $C_p$ are fixed, the harmonic filter $Y_j(\theta)$ carries angular dependence, and an invariant radial network supplies reduced coefficients $w_p(\lVert r_{xy}\rVert,r_z)$:

$$z_o=\sum_p w_p(\lVert r_{xy}\rVert,r_z)(C_p)_{oij}x_iY_j(\theta).$$

The hidden regular representations admit coordinatewise `PointActiv`. Both Wigner--Eckart layers reuse the same externally computed harmonic sample but have separate radial networks.

In [ ]:
class ExternalHarmonicNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.harmonics = harmonics
        self.input_layer = nn.WETensorProduct(
            input_rep, filter_rep, hidden_rep, shared_weights=False
        )
        self.activation = nn.PointActiv(hidden_rep, torch.relu)
        self.output_layer = nn.WETensorProduct(
            hidden_rep, filter_rep, output_rep, shared_weights=False
        )
        # These networks are shared across samples. Their outputs are the
        # external reduced coefficients expected by WETensorProduct.
        self.radial_in = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.input_layer.weight_numel),
        )
        self.radial_out = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.output_layer.weight_numel),
        )

    def forward(self, features, edge_displacements):
        theta = torch.atan2(edge_displacements[..., 1], edge_displacements[..., 0])
        filter_features = nn.RepresentationTensor(self.harmonics(theta), filter_rep)
        invariant_geometry = torch.stack(
            (torch.linalg.vector_norm(edge_displacements[..., :2], dim=-1), edge_displacements[..., 2]),
            dim=-1,
        )
        w_in = self.radial_in(invariant_geometry)
        w_out = self.radial_out(invariant_geometry)
        h_pre = self.input_layer(features, filter_features, w_in)
        h = self.activation(h_pre)
        y = self.output_layer(h, filter_features, w_out)
        return y, filter_features, w_in, w_out, h_pre, h

model = ExternalHarmonicNetwork().eval()
y, Y_typed, w_in, w_out, h_pre, h = model(x, edge_displacement)
print('input/output reduced-weight counts:', model.input_layer.weight_numel, model.output_layer.weight_numel)
print('hidden before PointActiv:', h_pre.tensor)
print('hidden after  PointActiv:', h.tensor)
print('physical output (vector, scalars):', unpack_output(y.tensor))
kernel_basis = model.input_layer.sample_kernel_basis(Y_typed)
print('sampled input-kernel basis shape [batch, paths, out, in]:', tuple(kernel_basis.shape))

## Step 4: rotate node features and edge geometry together

Equivariance requires rotating both the node features and the edge displacement $r_{ij}$. This displacement is filter geometry, not an additional node feature. The harmonic values generally change, while radial weights remain identical because their inputs are invariant.

In [ ]:
errors = []
for k, element in enumerate(G.elements):
    x_k = x.transform_fibers(element)
    edge_k = rotate_points(edge_displacement, element)
    y_k, Y_k, w_in_k, w_out_k, _, _ = model(x_k, edge_k)
    expected_k = y.transform_fibers(element)
    error = (y_k.tensor - expected_k.tensor).abs().max().item()
    errors.append(error)
    torch.testing.assert_close(y_k.tensor, expected_k.tensor, atol=5e-5, rtol=5e-5)
    torch.testing.assert_close(w_in_k, w_in, atol=1e-6, rtol=1e-6)
    torch.testing.assert_close(w_out_k, w_out, atol=1e-6, rtol=1e-6)
    in_vectors_k, in_scalars_k = unpack_input(x_k.tensor)
    out_vector_k, out_scalars_k = unpack_output(y_k.tensor)
    print(f'rotation {k}: angle={60*k:3d} degrees')
    print('  rotated edge r_ij:', edge_k[0].tolist())
    print('  circular Y      :', Y_k.tensor[0].tolist())
    print('  input vectors   :', in_vectors_k[0].tolist())
    print('  input scalars   :', in_scalars_k[0].tolist())
    print('  output vector   :', out_vector_k[0].tolist())
    print('  output scalars  :', out_scalars_k[0].tolist())
    print(f'  max equivariance error: {error:.3e}')

print('maximum over all rotations:', max(errors))

## Step 5: follow the harmonic filter into the regular hidden state

The left panel samples the circular filter only at the six C6-related edge orientations. The right panel shows the resulting activated regular hidden state. Each group action changes the harmonic components and cyclically permutes both six-coordinate regular fields.

In [ ]:
import matplotlib.pyplot as plt

hidden_by_rotation, harmonic_by_rotation = [], []
output_vectors, output_scalars = [], []
for element in G.elements:
    x_k = x.transform_fibers(element)
    edge_k = rotate_points(edge_displacement, element)
    y_k, Y_k, _, _, _, h_k = model(x_k, edge_k)
    vector_k, scalars_k = unpack_output(y_k.tensor)
    hidden_by_rotation.append(h_k.tensor[0].detach())
    harmonic_by_rotation.append(Y_k.tensor[0].detach())
    output_vectors.append(vector_k[0].detach())
    output_scalars.append(scalars_k[0].detach())
hidden_by_rotation = torch.stack(hidden_by_rotation).cpu()
harmonic_by_rotation = torch.stack(harmonic_by_rotation).cpu()
output_vectors = torch.stack(output_vectors).cpu()
output_scalars = torch.stack(output_scalars).cpu()
angles_deg = torch.arange(6) * 60
colors = plt.cm.hsv(torch.linspace(0, 5/6, 6).numpy())

harmonic_labels = []
for frequency, mode in harmonics._layout:
    if mode == 'pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'sin({frequency}θ)'))
    elif mode == 'conjugate_pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'-sin({frequency}θ)'))
    else:
        harmonic_labels.append(f'{mode}({frequency}θ)')

fig_state, (ax_harm, ax_hidden) = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
for component, label in enumerate(harmonic_labels):
    ax_harm.plot(angles_deg, harmonic_by_rotation[:, component], marker='o', label=label)
ax_harm.set(xticks=angles_deg.tolist(), xlabel='C6 rotation', ylabel='harmonic value', title='Circular harmonics: frequencies 0, 1, 2, 3')
ax_harm.grid(alpha=0.3); ax_harm.legend(fontsize=7, ncols=2)

image = ax_hidden.imshow(hidden_by_rotation, aspect='auto', cmap='coolwarm')
ax_hidden.axvline(5.5, color='white', linewidth=2)
ax_hidden.set(xticks=range(12), yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='regular coordinate (copies 1 | 2)', ylabel='rotation', title='Hidden 2 Reg(C6) after PointActiv')
fig_state.colorbar(image, ax=ax_hidden, shrink=0.75)
plt.show()

## Step 6: inspect the Wigner--Eckart kernel itself

`sample_kernel_basis` returns one matrix $K_p(r)$ per allowed coupling path. The invariant radial network supplies coefficients $w_p(r)$. Contracting them produces the effective matrix $K(r)=\sum_p w_p(r)K_p(r)$, and the assertion below verifies the concrete operation $h_{pre}=K(r)x$.

In [ ]:
basis_at_edge = model.input_layer.sample_kernel_basis(Y_typed)[0].detach()
effective_kernel = torch.einsum('p,poi->oi', w_in[0].detach(), basis_at_edge).cpu()
torch.testing.assert_close(h_pre.tensor[0], effective_kernel @ x.tensor[0], atol=2e-5, rtol=2e-5)

fig_kernel, ax_kernel = plt.subplots(figsize=(7, 5), constrained_layout=True)
kernel_image = ax_kernel.imshow(effective_kernel, aspect='auto', cmap='coolwarm')
ax_kernel.set(xlabel='input coordinate', ylabel='hidden coordinate', title='Effective input-layer kernel K(r)')
fig_kernel.colorbar(kernel_image, ax=ax_kernel, shrink=0.75)
plt.show()

## Step 7: unpack the final tensor-product output

The final layer produces one $xy$ vector irrep and four trivial coordinates. We combine the first trivial coordinate with $xy$ to reconstruct the physical 3D vector; the remaining three are displayed as scalar channels.

In [ ]:
fig_output = plt.figure(figsize=(12, 5), constrained_layout=True)
ax_vec = fig_output.add_subplot(1, 2, 1, projection='3d')
for k, (vector, color) in enumerate(zip(output_vectors, colors)):
    ax_vec.quiver(0, 0, 0, *vector.tolist(), color=color, linewidth=2, label=f'{60*k}°')
ax_vec.set(xlabel='x', ylabel='y', zlabel='z', title='WETensorProduct output vector')
output_limit = max(1e-3, 1.15 * output_vectors.abs().max().item())
ax_vec.set_xlim(-output_limit, output_limit); ax_vec.set_ylim(-output_limit, output_limit); ax_vec.set_zlim(-output_limit, output_limit); ax_vec.set_box_aspect((1, 1, 1))
ax_vec.legend(ncols=2, fontsize=7)

ax_scalar = fig_output.add_subplot(1, 2, 2)
for channel in range(3):
    ax_scalar.plot(angles_deg, output_scalars[:, channel], marker='o', label=f'output scalar {channel+1}')
ax_scalar.set(xticks=angles_deg.tolist(), xlabel='C6 rotation', ylabel='value', title='WETensorProduct output scalars')
ax_scalar.grid(alpha=0.3); ax_scalar.legend()
plt.show()